### Running PageIndex Locally

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.home() / "Projects" / "RAG_v1" / ".env")
NVIDIA_API_KEY = os.environ["NVIDIA_API_KEY"]

import pageindex.utils as _u

# 1) Don't hammer NVIDIA's free tier → no 429s
_u.SUMMARY_CONCURRENCY = 2

# 2) Cap oversized summary prompts (PORT_PACKAGE has a 200k-char node) → no 504s
_orig_acomp = _u.llm_acompletion


async def _capped(model, prompt):
    if len(prompt) > 6000:
        prompt = prompt[:6000] + "\n...[truncated]"
    return await _orig_acomp(model, prompt)


_u.llm_acompletion = _capped

from pageindex import PageIndexLocalClient, utils

pi_client = PageIndexLocalClient(
    model="nvidia_nim/meta/llama-3.1-8b-instruct",
    summary_model="nvidia_nim/meta/llama-3.1-8b-instruct",
    retrieve_model="nvidia_nim/meta/llama-3.1-8b-instruct",
    storage_path=".pageindex",
)
print(pi_client.__class__.__name__)


PageIndexLocalClient


### Building The JSON Tree

In [3]:
PDF_SOURCE_DIR = "Policy Documents Curated 15"
REGISTRY_PATH = "pdf_registry.json"

pdf_files = sorted(
    os.path.join(PDF_SOURCE_DIR, f)
    for f in os.listdir(PDF_SOURCE_DIR)
    if f.lower().endswith(".pdf")
)
print(f"Found {len(pdf_files)} PDFs")

existing = {d["name"]: d["id"] for d in pi_client.list_documents()["documents"]}
doc_ids = {}
for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path)
    if filename in existing:
        doc_ids[filename] = existing[filename]
        print(f"⏭️  Already indexed: {filename}")
        continue
    doc_ids[filename] = pi_client.submit_document(pdf_path)["doc_id"]  # synchronous
    print(f"✅ Submitted: {filename} → {doc_ids[filename]}")

print("\nAll doc_ids:", doc_ids)

Found 15 PDFs
⏭️  Already indexed: 14..Trade credit insc_GEN756.pdf
⏭️  Already indexed: Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf
⏭️  Already indexed: Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf
⏭️  Already indexed: Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf
⏭️  Already indexed: Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf
⏭️  Already indexed: Cyber_Shield_Policy_Wordings_78baa23b5a.pdf
⏭️  Already indexed: PORT_PACKAGE_Policy_wording_a5f317091b.pdf
⏭️  Already indexed: Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf
⏭️  Already indexed: Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf
⏭️  Already indexed: Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf
⏭️  Already indexed: SBI_General_Livestock_Policy_Wording_38ef0b201b.pdf
⏭️  Already indexed: Weather_Insurance_Policy_Wordings_Retail_6a1bd806a7.pdf
⏭️  Already indexed: hdfc-life-smart-pension-plus-v13

In [4]:
import json

print(json.dumps(doc_ids, indent=4))

{
    "14..Trade credit insc_GEN756.pdf": "pi-58068b94fd464bedaf251e3b442d3183",
    "Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf": "pi-7a77210c58344134988c513ce697a93a",
    "Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf": "pi-66af3e82285e4e24819dcc87fcda1343",
    "Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf": "pi-a47f79df4ac040f5984d512c4e1225ae",
    "Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf": "pi-12de45f887e644009982f039d294eb7e",
    "Cyber_Shield_Policy_Wordings_78baa23b5a.pdf": "pi-6392aba4cb2f45ce8abf2f75871f3e93",
    "PORT_PACKAGE_Policy_wording_a5f317091b.pdf": "pi-da50dbffde964da4ba36ede6c7079e11",
    "Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf": "pi-155f96f798b24b41982378c2bef596a5",
    "Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf": "pi-9d8ead02fe1f4f6284671bf58c07e80b",
    "Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf": "pi-

In [5]:
d = pi_client.list_documents()["documents"][0]["id"]
t = pi_client.get_tree(d, node_summary=True)["result"]
print(len(t), "top-level nodes")
print(utils.create_node_mapping(t).keys().__len__(), "total nodes")


66 top-level nodes
72 total nodes


### Creating PDF registry for routing

In [6]:
import json

registry = {
    d["id"]: {"doc_id": d["id"], "filename": d["name"], "description": d["description"]}
    for d in pi_client.list_documents()["documents"]
}
with open("pdf_registry.json", "w") as f:
    json.dump(registry, f, indent=2)
print(f"Registry written: {len(registry)} docs, 0 NIM calls")


Registry written: 15 docs, 0 NIM calls


### Calling Nim API

In [7]:
from openai import OpenAI


def call_nim(
    prompt,
    model: str,
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
    temperature=0,
    max_tokens=1024,
):
    client = OpenAI(base_url=base_url, api_key=api_key)
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message.content

In [8]:
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

models = client.models.list()
for model in models:
    print(model.id)


01-ai/yi-large
adept/fuyu-8b
ai21labs/jamba-1.5-large-instruct
aisingapore/sea-lion-7b-instruct
baai/bge-m3
bigcode/starcoder2-15b
databricks/dbrx-instruct
deepseek-ai/deepseek-coder-6.7b-instruct
deepseek-ai/deepseek-v4-flash-0731
google/codegemma-1.1-7b
google/codegemma-7b
google/deplot
google/diffusiongemma-26b-a4b-it
google/gemma-2b
google/gemma-3-12b-it
google/gemma-3-4b-it
google/gemma-4-31b-it
google/recurrentgemma-2b
ibm/granite-3.0-3b-a800m-instruct
ibm/granite-3.0-8b-instruct
ibm/granite-34b-code-instruct
ibm/granite-8b-code-instruct
meta/codellama-70b
meta/llama-3.1-70b-instruct
meta/llama-3.1-8b-instruct
meta/llama-3.2-11b-vision-instruct
meta/llama-3.2-1b-instruct
meta/llama-3.2-3b-instruct
meta/llama-3.2-90b-vision-instruct
meta/llama-3.3-70b-instruct
meta/llama-guard-4-12b
meta/llama2-70b
meta/muse-glimmer-30b
microsoft/kosmos-2
microsoft/phi-3-vision-128k-instruct
microsoft/phi-3.5-moe-instruct
minimaxai/minimax-m3
mistralai/codestral-22b-instruct-v0.1
mistralai/mistral

In [9]:
response = call_nim(
    prompt="What is the capital of India?", model="meta/llama-3.1-8b-instruct"
)
print(response)

The capital of India is New Delhi.


### Query Router

In [10]:
def route_query(query: str) -> dict:
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        registry = json.load(f)

    catalog = [
        {
            "doc_id": v["doc_id"],
            "filename": v["filename"],
            "description": v["description"],
        }
        for v in registry.values()
    ]

    prompt = f"""You are a document router for an insurance Q&A assistant.
A user has asked a question. Decide which documents to search and classify the question type.

User question: {query}

Available documents:
{json.dumps(catalog, indent=2)}

Classify the question into ONE of these types:
- "general"    : conceptual, definitional, or comparative questions ("what is X?", "explain Y", "how does Z work", "difference between A and B"). Answerable from general knowledge — EVEN IF one of the documents happens to cover that topic.
- "specific"   : the user asks about a concrete provision, condition, claim, or coverage detail of a policy — the kind of thing you'd only find in the actual wording (grace periods, exclusions, claim process, deductibles, coverage limits).
- "ambiguous"  : too vague to route (e.g. "tell me about my policy").

Reply ONLY with this JSON, nothing else:
{{
    "thinking": "<your reasoning>",
    "type": "specific",
    "doc_ids": ["doc_id_1"],
    "clarification": ""
}}

Rules:
- type "general"   → doc_ids must be [], clarification must be ""
- type "ambiguous" → doc_ids must be [], clarification must be a follow-up question
- type "specific"  → clarification must be ""
- Rule of thumb: if the answer wouldn't change between documents, it's "general". Only search documents when the user needs THEIR policy's actual terms.
- For "specific": list at most the 3 most relevant doc_ids, never more.
"""

    result = call_nim(prompt, model="meta/llama-3.1-8b-instruct")
    if result is None:
        raise ValueError("call_nim returned no response")
    parsed = json.loads(result)

    print(f"🔍 Type     : {parsed['type']}")
    print(f"📄 Doc IDs  : {parsed['doc_ids']}")
    print(f"💭 Reasoning: {parsed['thinking']}")

    return parsed


In [11]:
# # Test all three types
# route_query("What happens if I miss a premium payment?")  # should be specific
# route_query("What is term insurance?")  # should be general
# route_query("Tell me about my policy")  # should be ambiguous


### Search nodes in relevant documents

In [12]:
def search_nodes(doc_id: str, query: str) -> list[str]:
    """
    Searches the node tree of a single PDF for content relevant to the query.
    Returns a list of text chunks tagged with source filename and page number.
    """
    if not pi_client.is_retrieval_ready(doc_id):
        print(f"⚠️  Doc {doc_id} not ready — skipping.")
        return []

    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        registry = json.load(f)

    filename = registry[doc_id]["filename"]

    # fetch tree for this specific doc
    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    prompt = f"""You are given a question and a tree structure of a document.
Each node contains a node id, title, and summary.
Find all nodes likely to contain the answer to the question.

Question: {query}

Document tree:
{json.dumps(tree_without_text, indent=2)}

Reply ONLY with this JSON:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

    response_text = nim_call(prompt, model="deepseek-ai/deepseek-v4-flash-0731")
    if not response_text:
        raise ValueError("nim_call returned no response after 3 retries")
    result = json.loads(response_text)
    
    node_map = utils.create_node_mapping(tree)

    chunks = []
    for node_id in result["node_list"]:
        if node_id not in node_map:
            continue
        node = node_map[node_id]
        chunks.append(
            f"[Source: {filename}, Page {node['page_index']}]\n{node['text'][:4000]}"
        )

    print(f"  📑 {filename}: {len(chunks)} relevant node(s) found")
    return chunks


In [13]:
def ask(query: str):
    print(f"\n{'=' * 60}")
    print(f"Query: {query}")
    print("=" * 60)

    # Step 1: route — classify query and get relevant doc_ids
    routing = route_query(query)
    q_type = routing["type"]

    # Step 2: branch based on question type
    if q_type == "ambiguous":
        print(f"\nCould you clarify: {routing['clarification']}")
        return

    elif q_type == "general":
        print("\nGeneral question — answering from LLM knowledge.\n")
        prompt = f"""Answer this insurance question in simple plain language.
        Start with a one-sentence summary. Avoid jargon.
        Question: {query}"""
        answer = call_nim(
            prompt,
            model="meta/llama-3.1-70b-instruct",
        )
        utils.print_wrapped(answer)

    elif q_type == "specific":
        print(f"\nSearching {len(routing['doc_ids'])} document(s)...")

        # Step 3: search nodes in each routed doc and collect chunks
        all_chunks = []
        for doc_id in routing["doc_ids"]:
            chunks = search_nodes(doc_id, query)
            all_chunks.extend(chunks)

        if not all_chunks:
            print("No relevant content found.")
            return

        context = "\n\n---\n\n".join(all_chunks)

        # Step 4: generate answer from retrieved context
        prompt = f"""Answer the question based only on the context below.
If context comes from multiple documents, mention which document each point is from.

Question: {query}

Context:
{context}

Instructions:
- Use plain simple language, avoid legal jargon
- Use "you" and "your" instead of "the policyholder"
- Start with a one-sentence summary
- End with "Bottom line:" telling the user what to actually do or know
"""
        answer = call_nim(prompt, model="meta/llama-3.1-70b-instruct")
        print("\n📝 Answer:\n")
        utils.print_wrapped(answer)

In [14]:
# ask("What happens if I miss a premium payment?")


In [15]:
# ask("What is the SECTION V Of CYBER VAULTEDGE policy wording?")

In [16]:
import json
import os
import pickle
import random
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
EVAL_API_KEY = os.getenv("EVAL_API_KEY")

# ── Single NIM client — used for EVERYTHING in this notebook ─────────────────
nim = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=EVAL_API_KEY,
)

GENERATOR_MODEL = "meta/llama-3.1-8b-instruct"  # for test-set generation
JUDGE_MODEL = "meta/llama-3.1-70b-instruct"  # for evaluation scoring
# ─────────────────────────────────────────────────────────────────────────────
# NOTE: If you have access to qwen/qwq-32b, set JUDGE_MODEL to that.
# A bigger judge = more reliable scores. Generator model can stay small.
# ─────────────────────────────────────────────────────────────────────────────

CACHE_DIR = Path("cache/")
CACHE_DIR.mkdir(exist_ok=True)


def nim_call(prompt: str, model: str = GENERATOR_MODEL, max_tokens: int = 512) -> str:
    """
    Single NIM API call with retry logic.
    Sleeps 1s between calls automatically to avoid 429s on free tier.
    """
    for attempt in range(3):
        try:
            time.sleep(1)  # Rate limit buffer — DO NOT REMOVE
            response = nim.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"  ⚠ Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    return ""  # Return empty string on total failure — handled downstream


print("✅ Setup complete.")


✅ Setup complete.


In [17]:
testset_file = CACHE_DIR / "eval_testset.json"
if testset_file.exists():
    with open(testset_file) as f:
        test_set = json.load(f) 
    print(f"✅ Loaded {len(test_set)} cached test pairs — skipping generation")


✅ Loaded 29 cached test pairs — skipping generation


In [18]:
# Preview the test set
print(f"Total questions: {len(test_set)}\n")
for i, item in enumerate(test_set[:5], 1):
    print(f"Q{i}: {item['question']}")
    print(f"   GT: {item['ground_truth'][:120]}...")
    print(f"   From: {item['source_file']} p.{item['page']}")
    print()


Total questions: 29

Q1: What is the secondary source of observed weather index referred to as in the policy?
   GT: The secondary source of observed weather index is referred to as the 'Backup data source/ Weather Station'....
   From: Weather_Insurance_Policy_Wordings_Retail_6a1bd806a7.pdf p.1

Q2: What is included in the definition of a Launch Vehicle?
   GT: A Launch Vehicle includes any vehicle, including parts detached en route, designed, constructed or intended to place int...
   From: Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf p.4

Q3: What is the IRDA of India Registration Number of the insurance company?
   GT: The IRDA of India Registration Number is 108....
   From: Weather_Insurance_Policy_Wordings_Retail_6a1bd806a7.pdf p.1

Q4: What is the process for assessing the market value of an animal to be insured?
   GT: The market Value of the animal to be insured will be assessed and agreed jointly by the Insurer and the policyholder....
   From: SBI_General_Livestock

In [39]:
TREE_CACHE = {}
NODE_MAP_CACHE = {}
REGISTRY_CACHE = None


def load_registry() -> dict:
    global REGISTRY_CACHE
    if REGISTRY_CACHE is None:
        with open(REGISTRY_PATH, encoding="utf-8") as f:
            REGISTRY_CACHE = json.load(f)
    return REGISTRY_CACHE


def doc_id_for_eval_item(item: dict, registry: dict) -> str | None:
    """Eval rows already know their source PDF, so avoid an LLM router call."""
    source_file = os.path.basename(item.get("source_file", ""))
    for doc_id, meta in registry.items():
        if os.path.basename(meta["filename"]) == source_file:
            return doc_id
    return None


def get_tree_and_node_map(doc_id: str):
    if doc_id not in TREE_CACHE:
        tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
        TREE_CACHE[doc_id] = tree
        NODE_MAP_CACHE[doc_id] = utils.create_node_mapping(tree)
    return TREE_CACHE[doc_id], NODE_MAP_CACHE[doc_id]


def search_nodes_cached(doc_id: str, query: str, max_chunks: int = 2) -> list[str]:
    """Same PageIndex tree selection as search_nodes(), but caches local tree work."""
    if not pi_client.is_retrieval_ready(doc_id):
        print(f"  Doc {doc_id} not ready - skipping.")
        return []

    registry = load_registry()
    filename = registry[doc_id]["filename"]
    tree, node_map = get_tree_and_node_map(doc_id)
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    prompt = f"""You are given a question and a tree structure of a document.
Each node contains a node id, title, and summary.
Find up to {max_chunks} nodes likely to contain the answer to the question.

Question: {query}

Document tree:
{json.dumps(tree_without_text, indent=2)}

Reply ONLY with this JSON:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

    response_text = nim_call(prompt, model="deepseek-ai/deepseek-v4-flash-0731", max_tokens=256)
    if not response_text:
        raise ValueError("nim_call returned no response after 3 retries")

    result = json.loads(response_text)
    chunks = []
    for node_id in result.get("node_list", [])[:max_chunks]:
        node = node_map.get(node_id)
        if not node:
            continue
        chunks.append(f"[Source: {filename}, Page {node['page_index']}]\n{node['text'][:3000]}")

    print(f"  {filename}: {len(chunks)} relevant node(s) found")
    return chunks


def run_pageindex_pipeline(
    test_set: list[dict],
    start: int = 5,
    limit: int | None = 15,
    max_chunks: int = 2,
    sleep_between_questions: float = 0.5,
) -> list[dict]:
    """
    Fast eval path:
    - uses each test row's source_file instead of the LLM router
    - searches only that PDF, matching ask()'s specific-document path
    - caches PageIndex trees/node maps
    - writes after each question so a batch can resume safely
    """
    registry = load_registry()
    end = len(test_set) if limit is None else min(start + limit, len(test_set))
    batch = test_set[start:end]
    outputs = []

    print(f"Running PageIndex RAG on questions {start + 1}-{end} of {len(test_set)}...\n")

    for offset, item in enumerate(batch, start + 1):
        q = item["question"]
        print(f"[{offset}/{len(test_set)}] {q[:70]}...")

        try:
            doc_id = doc_id_for_eval_item(item, registry)
            q_type = "eval_source_file"

            if doc_id is None:
                routing = route_query(q)
                q_type = routing["type"]
                doc_ids_to_search = routing.get("doc_ids", [])[:1]
            else:
                doc_ids_to_search = [doc_id]

            all_chunks = []
            for doc_id_to_search in doc_ids_to_search:
                all_chunks.extend(search_nodes_cached(doc_id_to_search, q, max_chunks=max_chunks))

            if all_chunks:
                context = "\n\n---\n\n".join(all_chunks)
                answer = nim_call(
                    f"""Answer the question based only on the context below.
Question: {q}

Context:
{context}

Instructions:
- Use plain simple language
- Start with a one-sentence summary
- End with "Bottom line:" telling the user what to know""",
                    model=GENERATOR_MODEL,
                    max_tokens=384,
                )
            else:
                answer = "No relevant content found."

            outputs.append(
                {
                    "question": q,
                    "ground_truth": item["ground_truth"],
                    "contexts": all_chunks,
                    "answer": answer,
                    "q_type": q_type,
                    "source_file": item["source_file"],
                    "doc_ids_searched": doc_ids_to_search,
                }
            )
            print(f"  OK {len(all_chunks)} chunks | Answer: {answer[:60]}...")

        except Exception as e:
            print(f"  ERROR: {e}")
            outputs.append(
                {
                    "question": q,
                    "ground_truth": item["ground_truth"],
                    "contexts": [],
                    "answer": f"ERROR: {e}",
                    "q_type": "ERROR",
                    "source_file": item["source_file"],
                    "doc_ids_searched": [],
                }
            )

        time.sleep(sleep_between_questions)

    return outputs


# Run a small batch by default to stay under NIM rate limits.
EVAL_BATCH_START = 20
EVAL_BATCH_SIZE = 9
EVAL_MAX_CHUNKS = 2

batch_end = min(EVAL_BATCH_START + EVAL_BATCH_SIZE, len(test_set))
# pageindex_outputs_file = CACHE_DIR / f"pageindex_outputs_{EVAL_BATCH_START}_{batch_end}.json"
pageindex_outputs_file = CACHE_DIR / "pageindex_outputs.json"

if pageindex_outputs_file.exists():
    with open(pageindex_outputs_file) as f:
        pageindex_outputs = json.load(f)
    print(f"Loaded cached PageIndex outputs ({len(pageindex_outputs)} entries): {pageindex_outputs_file}")
else:
    pageindex_outputs = run_pageindex_pipeline(
        test_set,
        start=EVAL_BATCH_START,
        limit=EVAL_BATCH_SIZE,
        max_chunks=EVAL_MAX_CHUNKS,
    )
    with open(pageindex_outputs_file, "w") as f:
        json.dump(pageindex_outputs, f, indent=2)
    print(f"Saved PageIndex outputs to {pageindex_outputs_file}")


Loaded cached PageIndex outputs (29 entries): cache/pageindex_outputs.json


In [20]:
# ── The 4 evaluation prompts ──────────────────────────────────────────────────
# Each returns JSON: {"score": float, "reasoning": str}

FAITHFULNESS_PROMPT = """You are an expert evaluator assessing whether an AI answer is faithful to its source context.

QUESTION: {question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

TASK:
1. List every factual claim made in the Generated Answer.
2. For each claim, determine if it is directly supported by the Retrieved Context.
3. Score = (number of supported claims) / (total claims). If there are no claims, score 1.0.

Respond in this EXACT JSON format only (no markdown, no extra text):
{{"score": 0.0, "reasoning": "claim 1: supported/not supported because... claim 2: ..."}}

Score must be between 0.0 and 1.0."""


ANSWER_RELEVANCY_PROMPT = """You are an expert evaluator assessing whether an AI answer is relevant to the question asked.

QUESTION: {question}

GENERATED ANSWER:
{answer}

TASK:
Score how directly and completely the answer addresses the question.
- 1.0 = answer directly addresses all parts of the question
- 0.7 = answer mostly relevant but misses a part or adds off-topic content
- 0.4 = answer is vaguely related but does not really answer the question
- 0.0 = answer is completely off-topic or refuses to answer

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "explanation of why this score was given"}}"""


CONTEXT_PRECISION_PROMPT = """You are an expert evaluator assessing the quality of retrieved context for a RAG system.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT CHUNKS:
{context_numbered}

TASK:
For each retrieved chunk, decide if it is relevant to answering the question (given what the ground truth says).
Score = (number of relevant chunks) / (total chunks).
If no chunks were retrieved, score is 0.0.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Chunk 1: relevant/not relevant because... Chunk 2: ..."}}"""


CONTEXT_RECALL_PROMPT = """You are an expert evaluator assessing whether a RAG system retrieved all necessary information.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT:
{context}

TASK:
1. List every key piece of information in the Ground Truth Answer.
2. For each key piece, check if it is present in the Retrieved Context.
3. Score = (pieces present in context) / (total key pieces).
If no context was retrieved, score is 0.0.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Key point 1: found/not found in context... Key point 2: ..."}}"""


print("✅ Evaluation prompts defined.")


✅ Evaluation prompts defined.


In [40]:
def safe_parse_score(raw: str) -> dict:
    """
    Parses NIM's JSON response robustly.
    Handles cases where the model adds markdown fences or extra text.
    Returns {"score": float, "reasoning": str} or a fallback.
    """
    try:
        clean = raw.replace("```json", "").replace("```", "").strip()
        # Find the JSON object even if there's trailing text
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start == -1 or end == 0:
            raise ValueError("No JSON object found")
        parsed = json.loads(clean[start:end])
        score = float(parsed.get("score", 0.0))
        score = max(0.0, min(1.0, score))  # Clamp to [0, 1]
        return {"score": score, "reasoning": parsed.get("reasoning", "")}
    except Exception as e:
        return {"score": 0.0, "reasoning": f"Parse error: {e} | Raw: {raw[:100]}"}


def evaluate_single(item: dict) -> dict:
    """
    Evaluates one (question, contexts, answer, ground_truth) entry.
    Makes 4 NIM calls sequentially with sleep between each.
    Returns scores + reasoning for all 4 metrics.
    """
    q = item["question"]
    a = item["answer"]
    gt = item["ground_truth"]
    ctx = item.get("contexts", [])

    # Build context strings for prompts
    context_joined = "\n\n".join(ctx) if ctx else "[No context retrieved]"
    context_numbered = (
        "\n\n".join(f"[Chunk {i + 1}]:\n{c}" for i, c in enumerate(ctx))
        if ctx
        else "[No context retrieved]"
    )

    scores = {}

    # 1. Faithfulness
    raw = nim_call(
        FAITHFULNESS_PROMPT.format(question=q, context=context_joined, answer=a),
        model=JUDGE_MODEL,
        max_tokens=400,
    )
    scores["faithfulness"] = safe_parse_score(raw)

    # 2. Answer Relevancy
    raw = nim_call(
        ANSWER_RELEVANCY_PROMPT.format(question=q, answer=a),
        model=JUDGE_MODEL,
        max_tokens=300,
    )
    scores["answer_relevancy"] = safe_parse_score(raw)

    # 3. Context Precision
    raw = nim_call(
        CONTEXT_PRECISION_PROMPT.format(
            question=q, ground_truth=gt, context_numbered=context_numbered
        ),
        model=JUDGE_MODEL,
        max_tokens=400,
    )
    scores["context_precision"] = safe_parse_score(raw)

    # 4. Context Recall
    raw = nim_call(
        CONTEXT_RECALL_PROMPT.format(
            question=q, ground_truth=gt, context=context_joined
        ),
        model=JUDGE_MODEL,
        max_tokens=400,
    )
    scores["context_recall"] = safe_parse_score(raw)

    return scores


def evaluate_pipeline(pipeline_outputs: list[dict], strategy_name: str) -> list[dict]:
    """
    Evaluates all outputs of one RAG strategy.
    Returns a list of per-question results with scores.
    """
    results = []
    total = len(pipeline_outputs)
    print(f"\n{'=' * 60}")
    print(
        f"Evaluating: {strategy_name} ({total} questions × 4 metrics = {total * 4} NIM calls)"
    )
    print(f"{'=' * 60}")

    for i, item in enumerate(pipeline_outputs, 1):
        print(f"\n[{i}/{total}] {item['question'][:65]}...")

        # Skip entries that errored during pipeline run
        if item["answer"].startswith("ERROR") or item["answer"] == "TODO":
            print("  ⏭ Skipped (pipeline error)")
            continue

        scores = evaluate_single(item)

        result = {
            "question": item["question"],
            "answer": item["answer"],
            "ground_truth": item["ground_truth"],
            "n_contexts": len(item.get("contexts", [])),
            "faithfulness": scores["faithfulness"]["score"],
            "answer_relevancy": scores["answer_relevancy"]["score"],
            "context_precision": scores["context_precision"]["score"],
            "context_recall": scores["context_recall"]["score"],
            "reasoning": scores,
        }
        results.append(result)

        # Print live scores
        print(
            f"  F={result['faithfulness']:.2f}  "
            f"AR={result['answer_relevancy']:.2f}  "
            f"CP={result['context_precision']:.2f}  "
            f"CR={result['context_recall']:.2f}"
        )

    print(f"\n✅ Done: {len(results)}/{total} questions evaluated")
    return results


print("✅ Evaluator functions defined.")


✅ Evaluator functions defined.


In [41]:
# ── Evaluate PageIndex RAG ────────────────────────────────────────────────────
pageindex_eval_file = CACHE_DIR / "pageindex_eval_results.json"

if pageindex_eval_file.exists():
    with open(pageindex_eval_file) as f:
        pageindex_eval_results = json.load(f)
    print(
        f"✅ Loaded cached PageIndex eval results ({len(pageindex_eval_results)} entries)"
    )
else:
    pageindex_eval_results = evaluate_pipeline(pageindex_outputs, "PageIndex RAG")
    with open(pageindex_eval_file, "w") as f:
        json.dump(pageindex_eval_results, f, indent=2)
    print(f"✅ Saved to {pageindex_eval_file}")



Evaluating: PageIndex RAG (29 questions × 4 metrics = 116 NIM calls)

[1/29] What is the secondary source of observed weather index referred t...
  F=0.75  AR=1.00  CP=1.00  CR=1.00

[2/29] What is included in the definition of a Launch Vehicle?...
  F=0.00  AR=0.40  CP=0.00  CR=0.00

[3/29] What is the IRDA of India Registration Number of the insurance co...
  F=1.00  AR=1.00  CP=1.00  CR=1.00

[4/29] What is the process for assessing the market value of an animal t...
  F=0.00  AR=0.40  CP=0.00  CR=0.00

[5/29] What type of equipment is specified in the Insured Assets or Equi...
  F=0.80  AR=1.00  CP=1.00  CR=0.50

[6/29] What is the process for destruction of the insured animal to be e...
  F=0.50  AR=0.40  CP=0.00  CR=0.00

[7/29] What types of damage are excluded from coverage under the 'Subsid...
  F=0.00  AR=1.00  CP=0.00  CR=0.00

[8/29] What is the maximum penalty that a person can be liable for if th...
  F=1.00  AR=1.00  CP=1.00  CR=1.00

[9/29] How is the Overdue Notificat